# Pandas Practice — NYC Taxi: groupby, merge, reshape

**Goal:** build real fluency with `groupby`, `merge`, and reshaping (`pivot_table` / `melt`) on the taxi data.

**Rules (the whole point):**
1. Write every line **yourself**. No AI, no copy-paste.
2. If you're stuck, use the **hint** in each cell, or check the pandas docs — not a chatbot.
3. After each one, ask out loud: *"could I explain this line in an interview?"*
4. Don't peek at answers — there aren't any here on purpose.

Run cells top-to-bottom. Each exercise = a markdown prompt + an empty code cell for you.

## Setup (this one's done for you — just run it)

Loads one month (~3M rows) so it stays light on RAM. If it feels slow, uncomment the `.sample(...)` line to work on a subset.

In [1]:
import pandas as pd

trips = pd.read_parquet('data/yellow_tripdata_2024-01.parquet')
# trips = trips.sample(n=500_000, random_state=42)   # uncomment to go lighter

zones = pd.read_csv('data/taxi_zone_lookup.csv')

print(trips.shape)
trips.head()

(2964624, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


---
# Section A — groupby & aggregation

### A1. Count trips per `payment_type`
Produce the number of trips for each `payment_type`, sorted from most to least.

*Hint:* `groupby` + `size()` (or `.value_counts()`), then `sort_values`.

In [5]:
# your code here
df1 = trips.groupby(by='payment_type').size().reset_index(name='number_of_trips')
print(df1.sort_values(by='number_of_trips', ascending=False))

   payment_type  number_of_trips
1             1          2319046
2             2           439191
0             0           140162
4             4            46628
3             3            19597


### A2. Average `fare_amount` and `total_amount` by `passenger_count`
One result table, grouped by `passenger_count`, with the **mean** of both columns side by side.

*Hint:* `groupby('passenger_count')[['fare_amount','total_amount']].mean()`.

In [ ]:
df2 = trips.groupby('passenger_count')[['fare_amount', 'total_amount']].mean()
print(df2)
#should i have added reset index here?



                 fare_amount  total_amount
passenger_count                           
0.0                17.075336     25.327817
1.0                17.557052     26.205230
2.0                20.171285     29.520660
3.0                20.041299     29.138309
4.0                21.833799     30.877267
5.0                17.511870     26.269129
6.0                17.228569     25.801183
7.0                45.411250     57.735000
8.0                81.390980     95.668039
9.0                11.400000     18.450000


### A3. Multiple aggregations at once, by pickup location
For each `PULocationID`, compute in a single call: **trip count**, **total revenue** (`sum` of `total_amount`), and **average tip** (`mean` of `tip_amount`). Give the output columns clear names.

*Hint:* `.agg(...)` with named aggregations, e.g. `.agg(trips=('total_amount','size'), revenue=('total_amount','sum'), avg_tip=('tip_amount','mean'))`.

In [14]:
# your code here
df3 = trips.groupby('PULocationID').agg(trip_count=('total_amount', 'size'), total_revenue=('total_amount', 'sum'), average_tip=('tip_amount', 'mean'))
print(df3.sort_values(by='trip_count', ascending=False))

              trip_count  total_revenue  average_tip
PULocationID                                        
132               145240    11121928.53     8.863502
161               143471     3369045.01     3.077630
237               142708     2776195.32     2.589349
236               136465     2729558.57     2.642405
162               106717     2441727.65     3.019698
...                  ...            ...          ...
111                    1          90.00     0.000000
44                     1         354.23    58.750000
105                    1          18.48     3.080000
204                    1          77.71     0.000000
109                    1         111.48    18.580000

[260 rows x 3 columns]


### A4. Derived column, then group by it
First create `trip_duration_min` = minutes between dropoff and pickup. Then create `pickup_hour` = the hour (0–23) of `tpep_pickup_datetime`. Finally show the **average trip duration by pickup hour**.

*Hints:* subtract the two datetime columns → a Timedelta; `.dt.total_seconds() / 60`. For the hour: `trips['tpep_pickup_datetime'].dt.hour`.

In [17]:
# your code here
trips['minutes_diff'] = (trips['tpep_dropoff_datetime']-trips['tpep_pickup_datetime']).dt.total_seconds()/60
trips['pickup_hour'] = trips['tpep_pickup_datetime'].dt.hour
df4 = trips.groupby('pickup_hour').agg(avg_duration=('minutes_diff', 'mean'))
print(df4)
	

             avg_duration
pickup_hour              
0               14.764411
1               13.871398
2               13.222601
3               13.259529
4               15.197924
5               16.671865
6               16.127445
7               15.471865
8               15.648664
9               15.690070
10              15.839174
11              15.658831
12              15.602929
13              16.311287
14              17.180444
15              17.443927
16              17.598886
17              16.648574
18              14.926221
19              14.394222
20              14.058199
21              14.086324
22              14.484264
23              14.866783


---
# Section B — merge (joins)

### B1. Attach pickup zone names
Look at `zones` (`LocationID`, `Borough`, `Zone`, `service_zone`). Merge it onto `trips` so each trip gains its **pickup** `Borough` and `Zone`. Join key: `trips.PULocationID` ↔ `zones.LocationID`.

*Hints:* `pd.merge(trips, zones, left_on='PULocationID', right_on='LocationID', how='left')`. Think about which join type avoids dropping trips with an unmatched zone.

In [ ]:
# your code here


### B2. Attach dropoff zone names too (watch the column clash)
Starting from your B1 result, merge `zones` **again** for the **dropoff** side (`DOLocationID` ↔ `LocationID`). Now you have pickup AND dropoff borough/zone — make sure the columns are clearly distinguishable (not `Borough_x` / `Borough_y` that you can't read).

*Hints:* use `suffixes=('_pickup','_dropoff')`, or rename the zone columns before the second merge. Drop the redundant `LocationID` columns after.

In [ ]:
# your code here


### B3. Group after the join
Using your merged frame, compute **trip count** and **average `fare_amount`** for each **pickup borough**, sorted by trip count descending.

*Hint:* this is just A-style groupby, but on the column you gained from the merge.

In [ ]:
# your code here


---
# Section C — reshape (pivot & melt)

### C1. Pivot: borough × payment_type
Build a table where **rows = pickup borough**, **columns = `payment_type`**, and **values = trip count**. Fill empty combinations with 0.

*Hint:* `pd.pivot_table(df, index='Borough_pickup', columns='payment_type', values='<any col>', aggfunc='size', fill_value=0)` — or `aggfunc='count'` on a non-null column.

In [ ]:
# your code here


### C2. Demand heatmap matrix: hour × day-of-week
Make a matrix with **rows = pickup hour (0–23)**, **columns = day of week**, **values = trip count**. (This is exactly the Tableau heatmap you'll build later — good to understand the shape now.)

*Hints:* you'll need `pickup_hour` (from A4) and a day-of-week column (`.dt.dayofweek` or `.dt.day_name()`). Then `pivot_table` with `aggfunc='size'`.

In [ ]:
# your code here


### C3. melt — go from wide back to long
Take your C1 pivot (borough × payment_type) and **melt** it back into a long/tidy frame with columns like `Borough`, `payment_type`, `trip_count`. Understand *why* long format is what BI tools and `groupby` prefer.

*Hints:* `reset_index()` first so `Borough` is a real column, then `pd.melt(..., id_vars='Borough', var_name='payment_type', value_name='trip_count')`.

In [ ]:
# your code here


---
# Section D — challenge (put it together)

### D1. Top 10 busiest pickup zones (by name)
Return the 10 pickup **zones** (the human-readable `Zone` name, not the ID) with the most trips. Requires a merge (for the name) + groupby + sort + head.

*Hint:* you already have a merged frame from B — reuse it.

In [ ]:
# your code here


### D2. Average tip % by pickup borough
Compute `tip_pct = tip_amount / fare_amount` per trip, then the **average tip %** per pickup borough. Watch out for divide-by-zero / negative fares — decide how to handle them and be ready to justify it.

*Hints:* filter to `fare_amount > 0` before dividing; `replace([np.inf, -np.inf], np.nan)` is another guard. Multiply by 100 if you want percent.

In [ ]:
# your code here


---
### Done?
When you finish, pick any 2–3 cells and **explain them to me out loud** (or in chat). If you can't explain a line, that's the one to redo. I'll review your code for correctness, style, and whether it's interview-defensible — I won't hand you the answers.